# transformer-inference-lab — Benchmarks (MHA vs GQA vs MQA)

Runs `benchmarks/latency.py`, `memory.py`, `throughput.py` against all three trained checkpoints. This is where the memory and latency axes of the memory–latency–quality triangle get real numbers.

**Before running:** attach BOTH `transformer-inference-lab-data` and `transformer-inference-lab-checkpoints` as Input (sidebar → Add Input). The checkpoints dataset now contains mha.pt, gqa.pt, mqa.pt as separate versions — cell 4 copies all three into place.

Quality reference (already measured, for context in the final summary):
- MHA (n_kv_head=8): val_loss 5.6314
- MQA (n_kv_head=1): val_loss 5.6856
- GQA (n_kv_head=2): val_loss 5.7637

## 1. GPU check + clone repo

In [ ]:
!nvidia-smi
!git clone https://github.com/modestesavadogo/transformer-inference-lab.git
%cd /kaggle/working/transformer-inference-lab
!git log --oneline -8

## 2. Install dependencies

In [ ]:
!pip install -r requirements.txt --quiet

## 3. Link the training data
Not strictly needed for benchmarks (they use random token ids as prompts, not real data), but kept for consistency with the training notebooks.

In [ ]:
!mkdir -p data
!ln -s /kaggle/input/datasets/modestesavadogomaths/transformer-inference-lab-data/train.bin data/train.bin
!ln -s /kaggle/input/datasets/modestesavadogomaths/transformer-inference-lab-data/val.bin data/val.bin

## 4. Copy all three checkpoints into place
Copy, not symlink — some benchmark runs may want to modify/re-save state, and Input mounts are read-only. Verify the mount path with find first if this cell fails — the exact path pattern has changed before.

In [ ]:
# !pip install kaggle --quiet --upgrade   # also fixes the outdated-version warning you saw
# !kaggle kernels output modestesavadogomaths/transformer-inference-mha-lab -p mha_output/
# !kaggle kernels output modestesavadogomaths/transformer-inference-lab-gqa-training -p gqa_output/

In [ ]:
# !find mha_output/ -name "*.pt"
# !find gqa_output/ -name "*.pt"

In [ ]:
!mkdir -p checkpoint_upload
!cp /kaggle/input/datasets/modestesavadogomaths/transformer-inference-lab-checkpoints/mha.pt checkpoint_upload/
!cp /kaggle/input/datasets/modestesavadogomaths/transformer-inference-lab-checkpoints/gqa.pt checkpoint_upload/
!cp /kaggle/input/datasets/modestesavadogomaths/transformer-inference-lab-checkpoints/mqa.pt checkpoint_upload/
!ls -lh checkpoint_upload/

In [ ]:
# import json
# metadata = {
#     "title": "transformer-inference-lab-checkpoints",
#     "id": "modestesavadogomaths/transformer-inference-lab-checkpoints",
#     "licenses": [{"name": "CC0-1.0"}]
# }
# with open("checkpoint_upload/dataset-metadata.json", "w") as f:
#     json.dump(metadata, f, indent=2)

In [ ]:
# !kaggle datasets version -p checkpoint_upload/ -m "all three checkpoints merged together" --dir-mode zip

In [ ]:
!kaggle datasets files modestesavadogomaths/transformer-inference-lab-checkpoints

## 5. Verify each checkpoint's n_kv_head before benchmarking
Same discipline as the MQA training notebook — confirm each checkpoint is actually the variant its filename claims, before spending GPU time benchmarking it.

In [ ]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"using device: {device}")

expected = {"mha": 8, "gqa": 2, "mqa": 1}
checkpoints = {}

for name, expected_kv in expected.items():
    ckpt = torch.load(f"checkpoint_upload/{name}.pt", map_location=device)
    actual_kv = ckpt["config"]["n_kv_head"]
    val_loss = ckpt.get("val_loss")
    print(f"{name}: n_kv_head={actual_kv}, val_loss={val_loss}")
    assert actual_kv == expected_kv, f"{name}.pt has n_kv_head={actual_kv}, expected {expected_kv}"
    checkpoints[name] = ckpt

print("\nall three checkpoints verified")

In [ ]:
from train import estimate_val_loss

model_cfg_dict = checkpoints["mha"]["config"]
from src.model.transformer import GPT, ModelConfig
model = GPT(ModelConfig(**model_cfg_dict)).to(device)
model.load_state_dict(checkpoints["mha"]["model_state_dict"])

mha_val_loss = estimate_val_loss(model, "data", 1024, 8, device, eval_iters=50)
print("MHA val_loss (recomputed):", mha_val_loss)

checkpoints["mha"]["val_loss"] = mha_val_loss
torch.save(checkpoints["mha"], "checkpoint_upload/mha.pt")
print("mha.pt updated with val_loss")

In [ ]:
!mkdir -p results/checkpoints
!cp checkpoint_upload/mha.pt checkpoint_upload/gqa.pt checkpoint_upload/mqa.pt results/checkpoints/
!ls -lh results/checkpoints/

## 6. Latency benchmark — all three variants, two context lengths
With KV cache (the realistic decoding setup). Context lengths 512 and 2048 to see whether the gap between variants changes with context size.

In [ ]:
!mkdir -p results/latency

# latency.py cell — max_new_tokens=128, so context_length must stay ≤ 896
for name, kv in [("mha", 8), ("gqa", 2), ("mqa", 1)]:
    for ctx in [256, 512, 768]:
        print(f"=== {name} (kv_heads={kv}), context={ctx} ===")
        !python -m benchmarks.latency \
            --attention {name} --kv-heads {kv} \
            --context-length {ctx} --max-new-tokens 128 \
            --checkpoint results/checkpoints/{name}.pt \
            --out results/latency/{name}_ctx{ctx}.json

## 7. Naive (no-cache) baseline — MHA only
For the 02/03 README sections: quantifies the KV cache speedup itself, not the attention-variant comparison. Only needed once, on any one checkpoint — MHA chosen as it's the reference variant.

In [ ]:
!python -m benchmarks.latency \
    --attention mha --kv-heads 8 \
    --context-length 512 --max-new-tokens 64 \
    --no-cache \
    --checkpoint results/checkpoints/mha.pt \
    --out results/latency/mha_nocache_ctx512.json

## 8. Memory benchmark — all three variants, sweep of context lengths
Confirms the theoretical KV cache formula against real GPU allocator numbers.

In [ ]:
!mkdir -p results/memory

# memory.py cell — default max_new_tokens=64, so context+64 must stay ≤1024
for name, kv in [("mha", 8), ("gqa", 2), ("mqa", 1)]:
    print(f"=== {name} (kv_heads={kv}) ===")
    !python -m benchmarks.memory \
        --attention {name} --kv-heads {kv} \
        --context-lengths 256,512,768 \
        --checkpoint results/checkpoints/{name}.pt \
        --out results/memory/{name}.json

## 9. Throughput benchmark — all three variants, batch size sweep
This is the one most likely to show OOM for MHA before GQA/MQA — that OOM point itself is a result, not a failure. If a cell errors here, that's expected behavior from the script (it catches OOM and stops the sweep), not a bug — check the JSON output for the oom:true entry.

In [ ]:
!mkdir -p results/throughput

for name, kv in [("mha", 8), ("gqa", 2), ("mqa", 1)]:
    print(f"=== {name} (kv_heads={kv}) ===")
    !python -m benchmarks.throughput \
        --attention {name} --kv-heads {kv} \
        --context-length 512 --max-new-tokens 64 \
        --checkpoint results/checkpoints/{name}.pt \
        --batch-sizes 1,4,8,16,32,64 \
        --out results/throughput/{name}.json

In [ ]:
!git add README.md

## 10. Summary table
Quick sanity read before generating plots — pulls the key numbers from each JSON into one table.

In [ ]:
import json as json_lib
from pathlib import Path

print(f"{'variant':<6} {'ctx':<6} {'mean_lat_ms':<14} {'kv_cache_bytes':<16}")
for name in ["mha", "gqa", "mqa"]:
    for ctx in [512, 2048]:
        p = Path(f"results/latency/{name}_ctx{ctx}.json")
        if p.exists():
            d = json_lib.loads(p.read_text())
            print(f"{name:<6} {ctx:<6} {d['mean_latency_ms']:<14.2f} {d['kv_cache_bytes']:<16}")

print()
print(f"{'variant':<6} {'ctx':<6} {'theoretical_bytes':<20} {'peak_gpu_bytes':<16}")
for name in ["mha", "gqa", "mqa"]:
    p = Path(f"results/memory/{name}.json")
    if p.exists():
        d = json_lib.loads(p.read_text())
        for r in d["results"]:
            print(f"{name:<6} {r['context_length']:<6} {r['theoretical_kv_cache_bytes']:<20} {r['peak_gpu_memory_bytes']:<16}")

print()
print(f"{'variant':<6} {'batch':<6} {'tokens_per_sec':<16} {'oom':<6}")
for name in ["mha", "gqa", "mqa"]:
    p = Path(f"results/throughput/{name}.json")
    if p.exists():
        d = json_lib.loads(p.read_text())
        for r in d["sweep"]:
            if r.get("oom"):
                print(f"{name:<6} {r['batch_size']:<6} {'--':<16} {'YES':<6}")
            else:
                print(f"{name:<6} {r['batch_size']:<6} {r['tokens_per_sec']:<16.2f} {'no':<6}")

## 11. Package results for download
results/latency, results/memory, results/throughput are small JSON files — these get committed to GitHub (unlike checkpoints, which stay gitignored). Zip them for easy download from the Output tab.

In [ ]:
!cd results && zip -r ../benchmark_results.zip latency/ memory/ throughput/
!ls -lh benchmark_results.zip